In [1]:
"""
Stage 3 (ES vs LS) — Feature Selection with SMOTE inside outer folds
======================================================================

Variant of stage3_FS_ESLS.py that applies SMOTE on outer-TRAIN folds
(post-residualization, pre-MI) before bagged LASSO stability selection.
Two SMOTE strategies run sequentially:

  (a) BALANCED   — minority class raised to majority class count
                   (sampling_strategy='auto')
  (b) IMBALANCED — both classes oversampled 2x while preserving the
                   original inter-class ratio
                   (sampling_strategy={0: 2*n_LS, 1: 2*n_ES})

Class encoding:
  POS_LABEL = "ES"  (majority, ~60 samples in full set)
  NEG_LABEL = "LS"  (minority, ~42 samples in full set)

LEAKAGE GUARDS (Pedregosa et al. 2011 sklearn docs §3.2; Chawla et al.
2002 SMOTE; Bischl et al. 2012 nested resampling):

  * SMOTE is fit ONLY on outer-train data of each fold.
  * Synthetic samples NEVER cross the outer-fold boundary.
  * Outer test fold is never SMOTE-touched (clean held-out evaluation).
  * MI and bagged LASSO see SMOTE-augmented outer-train; this is the
    correct point of injection because both downstream steps benefit
    from the rebalanced distribution.

Pipeline per outer fold:
  A. Mirror OLS residualization (fit on outer-train, transform applied
     to outer-train; outer-test would be transformed identically if used)
  B. Variance pre-screen — top-5000 by variance on residualized
     outer-train
  C. SMOTE on residualized variance-screened outer-train
  D. MI ranking on SMOTE-augmented outer-train — top-1000
  E. Bagged LASSO stability on MI-top-1000 SMOTE-augmented data
  F. Selection frequency >= pi_b -> fold panel

Outer folds: 10 stratified x 3 seeds = 30 outer folds (per variant).
Inner CV: not used here (pi_b fixed at 0.6 — see efficiency notes
in stage3_FS_ESLS.py).

Outputs (per variant):
    Outputs/stage3_FS_ESLS_smote_balanced/
    Outputs/stage3_FS_ESLS_smote_imbalanced/
        consensus_panel.csv, fold_inclusion_per_gene.csv,
        fold_panels.json, fold_summary.csv,
        mi_inclusion_per_gene.csv, phase1_zero_variance_filter.csv,
        stability_path.csv, report.txt,
        smote_per_fold_counts.csv
"""

from __future__ import annotations

import json
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from joblib import Parallel, delayed
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline\ES vs LS")
SPLIT_DIR    = PIPELINE_DIR / "Outputs" / "stage2_split_stratum_aware_ESLS"
INPUT_TRAIN  = SPLIT_DIR / "train.csv"
META_TRAIN   = SPLIT_DIR / "meta_train.csv"
OUTPUT_BASE  = PIPELINE_DIR / "Outputs"

CONDITION_COL   = "condition"
CONFOUNDER_COLS = ["sex", "cell_type"]
SEEDS           = [42, 123, 256]

NEG_LABEL = "LS"   # 0  (minority)
POS_LABEL = "ES"   # 1  (majority)

SMOTE_VARIANTS = ["balanced", "imbalanced"]


@dataclass
class Config:
    n_outer:        int        = 10
    vst_round:      int        = 4
    var_prefilter_k: int       = 5000
    mi_top_k:        int       = 1000
    B_bootstraps:    int       = 50
    boot_frac:       float     = 0.5
    lambda_grid:     np.ndarray = field(default_factory=lambda:
                                        np.logspace(-2, 0.3, 3))
    pi_b_default:    float     = 0.6
    lasso_max_iter:  int       = 1000
    lasso_tol:       float     = 1e-2
    consensus_threshold: float = 0.80
    path_thresholds: List[float] = field(default_factory=lambda:
                                          [0.60, 0.70, 0.80, 0.90, 0.95])
    smote_k_neighbors: int     = 5
    n_jobs:          int       = -1


# ------------------------------ I/O helpers ---------------------------------
def load_inputs() -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, List[str]]:
    """
    Loads stage-2 train.csv (genes-as-rows, samples-as-columns) plus the
    aligned metadata. Robust to accidental transposition.
    """
    df = pd.read_csv(INPUT_TRAIN, index_col=0)
    meta = pd.read_csv(META_TRAIN, index_col=0)
    n_samples_meta = len(meta)
    if df.shape[0] == n_samples_meta and df.shape[1] != n_samples_meta:
        df = df.T

    gene_names = df.index.tolist()
    sample_ids = df.columns.tolist()
    meta = meta.loc[sample_ids]
    X = df.T.values.astype(np.float64)
    y = meta[CONDITION_COL].values
    return X, y, meta, gene_names


# --------------------------- Phase 1 helpers --------------------------------
def drop_zero_variance(X, gene_names):
    var = X.var(axis=0)
    keep_mask = var > 0
    return (X[:, keep_mask], keep_mask,
            [g for g, k in zip(gene_names, keep_mask) if k])


def round_vst(X, decimals=4):
    return np.round(X, decimals=decimals)


# --------------------- Phase 2A: mirror OLS residualization -----------------
def fit_mirror_ols(X_train, meta_train, conf_cols):
    enc = OneHotEncoder(drop="first", sparse_output=False,
                        handle_unknown="ignore")
    enc.fit(meta_train[conf_cols])

    def design(meta_block):
        D = enc.transform(meta_block[conf_cols])
        return np.column_stack([np.ones(len(meta_block)), D])

    C_train = design(meta_train)
    B_ols = np.linalg.pinv(C_train) @ X_train

    def residualize(X, meta_block):
        C = design(meta_block)
        return X - C[:, 1:] @ B_ols[1:, :]

    return residualize, B_ols


# --------------------- Phase 2B & 2D: variance + MI ranking -----------------
def variance_prescreen(X, k):
    if k >= X.shape[1]:
        return np.ones(X.shape[1], dtype=bool)
    var = X.var(axis=0)
    keep = np.zeros(X.shape[1], dtype=bool)
    keep[np.argsort(var)[-k:]] = True
    return keep


def mi_topk_filter(X, y_int, k, seed):
    mi = mutual_info_classif(X, y_int, random_state=seed,
                             discrete_features=False, n_neighbors=3)
    n = X.shape[1]
    keep = np.zeros(n, dtype=bool)
    if k >= n:
        keep[:] = True
    else:
        keep[np.argsort(mi)[-k:]] = True
    return keep, mi


# ----------------- Phase 2C: SMOTE inside outer fold ------------------------
def to_int(y_str):
    return np.where(np.asarray(y_str) == POS_LABEL, 1, 0).astype(int)


def smote_strategy(y_int, variant):
    """
    Returns the imblearn sampling_strategy and a label string.

      balanced   -> minority lifted to match majority (sklearn 'auto')
      imbalanced -> EVERY class doubled (preserves original ratio)
    """
    if variant == "balanced":
        return "auto", "balanced (minority -> majority)"
    if variant == "imbalanced":
        counts = pd.Series(y_int).value_counts().to_dict()
        target = {int(c): int(2 * n) for c, n in counts.items()}
        label = ("imbalanced (2x each: " +
                 ", ".join(f"{NEG_LABEL if c==0 else POS_LABEL}={n}->{2*n}"
                           for c, n in counts.items()) + ")")
        return target, label
    raise ValueError(f"Unknown SMOTE variant: {variant}")


def apply_smote(X, y_int, variant, seed, k_neighbors):
    """Fit-and-transform SMOTE. Returns (X_res, y_int_res, info dict)."""
    n_pos = int((y_int == 1).sum())
    n_neg = int((y_int == 0).sum())
    minority = min(n_pos, n_neg)
    k = min(k_neighbors, max(1, minority - 1))

    strat, label = smote_strategy(y_int, variant)
    sm = SMOTE(sampling_strategy=strat, random_state=seed, k_neighbors=k)
    X_res, y_res = sm.fit_resample(X, y_int)
    info = {
        "variant":         variant,
        "label":           label,
        "k_neighbors":     k,
        "n_pre_LS":        n_neg,
        "n_pre_ES":        n_pos,
        "n_post_LS":       int((y_res == 0).sum()),
        "n_post_ES":       int((y_res == 1).sum()),
        "n_synth_LS":      int((y_res == 0).sum() - n_neg),
        "n_synth_ES":      int((y_res == 1).sum() - n_pos),
    }
    return X_res, y_res, info


# ------------------- Phase 2E: Bagged LASSO stability selection -------------
def _fit_one_bootstrap(X, y_int, lambda_grid, boot_frac, seed,
                       max_iter, tol):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.choice(n, size=int(np.ceil(boot_frac * n)), replace=False)
    Xb, yb = X[idx], y_int[idx]
    if len(np.unique(yb)) < 2:
        return np.zeros((X.shape[1], len(lambda_grid)), dtype=bool)
    order = np.argsort(lambda_grid)[::-1]
    sel = np.zeros((X.shape[1], len(lambda_grid)), dtype=bool)
    clf = LogisticRegression(penalty="l1", solver="saga",
                              multi_class="multinomial", warm_start=True,
                              max_iter=max_iter, tol=tol, random_state=seed)
    for li in order:
        lam = lambda_grid[li]
        clf.C = 1.0 / max(lam, 1e-8)
        try:
            clf.fit(Xb, yb)
        except Exception:
            continue
        sel[:, li] = (np.abs(clf.coef_) > 1e-10).any(axis=0)
    return sel


def bagged_lasso_stability(X, y_int, B, lambda_grid, boot_frac, n_jobs,
                           seed, max_iter, tol):
    seeds = np.random.SeedSequence(seed).generate_state(B)
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_fit_one_bootstrap)(X, y_int, lambda_grid, boot_frac,
                                     int(s), max_iter, tol)
        for s in seeds
    )
    sel_stack = np.stack(results, axis=0)
    return sel_stack.mean(axis=0).max(axis=1)


def panel_at_threshold(pi_max, gene_names, threshold):
    return [g for g, p in zip(gene_names, pi_max) if p >= threshold]


# ----------------------------- Outer fold core ------------------------------
def run_outer_fold(X, y_str, meta, gene_names, train_idx,
                   fold_id, seed, smote_variant, cfg):
    X_tr     = X[train_idx]
    y_tr_str = y_str[train_idx]
    m_tr     = meta.iloc[train_idx]

    # A. Mirror OLS residualization
    residualize, _ = fit_mirror_ols(X_tr, m_tr, CONFOUNDER_COLS)
    X_tr_res = residualize(X_tr, m_tr)

    # B. Variance pre-screen
    var_keep = variance_prescreen(X_tr_res, cfg.var_prefilter_k)
    X_tr_var = X_tr_res[:, var_keep]
    var_genes = [g for g, k in zip(gene_names, var_keep) if k]
    n_var = int(var_keep.sum())

    # C. SMOTE on residualized variance-screened outer-train
    y_tr_int = to_int(y_tr_str)
    X_tr_var_sm, y_tr_int_sm, smote_info = apply_smote(
        X_tr_var, y_tr_int, smote_variant, seed, cfg.smote_k_neighbors)

    # D. MI on SMOTE-augmented outer-train
    mi_keep, _ = mi_topk_filter(X_tr_var_sm, y_tr_int_sm,
                                 cfg.mi_top_k, seed=seed)
    X_tr_mi = X_tr_var_sm[:, mi_keep]
    mi_genes = [g for g, k in zip(var_genes, mi_keep) if k]
    n_mi = int(mi_keep.sum())

    # E. Bagged LASSO stability on SMOTE-augmented MI top-K
    pi_max = bagged_lasso_stability(
        X_tr_mi, y_tr_int_sm, B=cfg.B_bootstraps,
        lambda_grid=cfg.lambda_grid, boot_frac=cfg.boot_frac,
        n_jobs=cfg.n_jobs, seed=seed,
        max_iter=cfg.lasso_max_iter, tol=cfg.lasso_tol)
    panel = panel_at_threshold(pi_max, mi_genes, cfg.pi_b_default)

    return {
        "fold_id":   fold_id,
        "n_var":     n_var,
        "n_mi":      n_mi,
        "best_pi_b": cfg.pi_b_default,
        "panel":     panel,
        "mi_genes":  mi_genes,
        "pi_max":    pi_max.tolist(),
        "smote":     smote_info,
    }


# ------------------------------ Variant runner ------------------------------
def run_variant(variant: str, X_phase1, y, meta, gene_names, n_dropped_zv,
                cfg: Config):
    out_dir = OUTPUT_BASE / f"stage3_FS_ESLS_smote_{variant}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"VARIANT: SMOTE = {variant.upper()}")
    print(f"Output dir: {out_dir.name}")
    print("=" * 72)

    pd.DataFrame({"gene": gene_names,
                  "kept_phase1": np.ones(len(gene_names), dtype=bool)}).to_csv(
        out_dir / "phase1_zero_variance_filter.csv", index=False)

    fold_results = []
    smote_count_rows = []
    t0 = time.time()
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=cfg.n_outer, shuffle=True,
                               random_state=seed)
        for k, (tr_idx, _) in enumerate(skf.split(X_phase1, y)):
            fold_id = f"seed{seed}_fold{k + 1}"
            t_fold = time.time()
            print(f"  [{fold_id}] n_train={len(tr_idx)}", end=" ", flush=True)
            res = run_outer_fold(
                X_phase1, y, meta, gene_names,
                train_idx=tr_idx, fold_id=fold_id,
                seed=seed * 1000 + k, smote_variant=variant, cfg=cfg)
            sm = res["smote"]
            smote_count_rows.append({
                "fold_id":     fold_id,
                "variant":     variant,
                "n_pre_LS":    sm["n_pre_LS"],
                "n_pre_ES":    sm["n_pre_ES"],
                "n_post_LS":   sm["n_post_LS"],
                "n_post_ES":   sm["n_post_ES"],
                "n_synth_LS":  sm["n_synth_LS"],
                "n_synth_ES":  sm["n_synth_ES"],
                "k_neighbors": sm["k_neighbors"],
            })
            elapsed = time.time() - t_fold
            print(f"-> SMOTE: LS {sm['n_pre_LS']}->{sm['n_post_LS']}, "
                  f"ES {sm['n_pre_ES']}->{sm['n_post_ES']}; "
                  f"var={res['n_var']}, MI={res['n_mi']}, "
                  f"panel={len(res['panel'])}  ({elapsed:.0f}s)")
            fold_results.append(res)
    total = time.time() - t0
    print(f"  Total Phase 2 time ({variant}): {total/60:.1f} min")

    pd.DataFrame(smote_count_rows).to_csv(
        out_dir / "smote_per_fold_counts.csv", index=False)

    # Phase 3 — consensus
    print(f"\n  Phase 3 ({variant}) — Consensus panel")
    n_folds = len(fold_results)
    inclusion = pd.Series(0, index=gene_names, dtype=float)
    for r in fold_results:
        for g in r["panel"]:
            inclusion[g] += 1
    inclusion /= n_folds

    path_rows = []
    for thr in cfg.path_thresholds:
        n = int((inclusion >= thr).sum())
        path_rows.append({"threshold": thr, "panel_size": n})
        print(f"    fold_inclusion >= {thr:.2f}  ->  panel size = {n}")
    pd.DataFrame(path_rows).to_csv(out_dir / "stability_path.csv", index=False)

    consensus = (inclusion[inclusion >= cfg.consensus_threshold]
                 .sort_values(ascending=False))
    print(f"  Final consensus panel (>= {cfg.consensus_threshold:.2f}): "
          f"{len(consensus)} genes")

    inclusion.sort_values(ascending=False).to_frame("fold_inclusion").to_csv(
        out_dir / "fold_inclusion_per_gene.csv")
    consensus.to_frame("fold_inclusion").to_csv(
        out_dir / "consensus_panel.csv")

    pd.DataFrame([{"fold_id": r["fold_id"], "n_var": r["n_var"],
                   "n_mi": r["n_mi"], "best_pi_b": r["best_pi_b"],
                   "panel_size": len(r["panel"])} for r in fold_results]
                 ).to_csv(out_dir / "fold_summary.csv", index=False)

    with open(out_dir / "fold_panels.json", "w") as f:
        json.dump([{"fold_id": r["fold_id"], "panel": r["panel"],
                    "best_pi_b": r["best_pi_b"], "n_mi": r["n_mi"],
                    "smote": r["smote"]} for r in fold_results], f, indent=2)

    mi_inclusion = pd.Series(0, index=gene_names, dtype=float)
    for r in fold_results:
        for g in r["mi_genes"]:
            mi_inclusion[g] += 1
    mi_inclusion /= n_folds
    mi_inclusion.sort_values(ascending=False).to_frame("mi_inclusion").to_csv(
        out_dir / "mi_inclusion_per_gene.csv")

    with open(out_dir / "report.txt", "w") as f:
        f.write(f"STAGE 3 (ES vs LS) SMOTE = {variant.upper()} — FS Report\n")
        f.write("=" * 60 + "\n")
        f.write(f"SMOTE variant       : {variant}\n")
        f.write(f"  pre-fold means    : LS={int(np.mean([r['n_pre_LS'] for r in smote_count_rows]))}, "
                f"ES={int(np.mean([r['n_pre_ES'] for r in smote_count_rows]))}\n")
        f.write(f"  post-fold means   : LS={int(np.mean([r['n_post_LS'] for r in smote_count_rows]))}, "
                f"ES={int(np.mean([r['n_post_ES'] for r in smote_count_rows]))}\n")
        f.write(f"Outer-fold seeds    : {SEEDS}\n")
        f.write(f"Outer folds (total) : {n_folds} ({cfg.n_outer} x {len(SEEDS)})\n")
        f.write(f"Variance pre-screen : top {cfg.var_prefilter_k}\n")
        f.write(f"MI top-K            : {cfg.mi_top_k}\n")
        f.write(f"B bootstraps        : {cfg.B_bootstraps}\n")
        f.write(f"pi_b                : {cfg.pi_b_default}\n")
        f.write(f"Total Phase 2 time  : {total/60:.1f} min\n\n")
        f.write("Stability path\n" + "-" * 30 + "\n")
        for row in path_rows:
            f.write(f"  >= {row['threshold']:.2f}  ->  "
                    f"{row['panel_size']} genes\n")
        f.write(f"\nConsensus threshold : {cfg.consensus_threshold:.2f}\n")
        f.write(f"Consensus panel size: {len(consensus)}\n")

    return out_dir, len(consensus)


# ----------------------------------- Main -----------------------------------
def main():
    cfg = Config()
    print("=" * 72)
    print("STAGE 3 (ES vs LS) SMOTE — Feature Selection (10 outer x 3 seeds)")
    print(f"Variants: {SMOTE_VARIANTS}")
    print(f"Class encoding: NEG={NEG_LABEL}=0, POS={POS_LABEL}=1")
    print(f"var_prefilter={cfg.var_prefilter_k}, MI top-K={cfg.mi_top_k}, "
          f"B={cfg.B_bootstraps}")
    print("=" * 72)

    X, y, meta, all_gene_names = load_inputs()
    print(f"Loaded train: {X.shape[0]} samples x {X.shape[1]} genes")
    print(f"Class balance: {pd.Series(y).value_counts().to_dict()}")

    # Phase 1 (shared across variants)
    X_phase1, kept_mask, gene_names = drop_zero_variance(X, all_gene_names)
    n_dropped_zv = int((~kept_mask).sum())
    X_phase1 = round_vst(X_phase1, decimals=cfg.vst_round)
    print(f"\nPhase 1 — zero-variance filter + round to {cfg.vst_round} decimals")
    print(f"  Dropped: {n_dropped_zv}; Kept: {X_phase1.shape[1]}")

    summary = []
    for variant in SMOTE_VARIANTS:
        out_dir, panel_size = run_variant(variant, X_phase1, y, meta,
                                           gene_names, n_dropped_zv, cfg)
        summary.append((variant, out_dir, panel_size))

    print("\n" + "=" * 72)
    print("SUMMARY")
    print("=" * 72)
    for variant, out_dir, panel_size in summary:
        print(f"  SMOTE={variant:<11} -> {out_dir.name}  "
              f"(consensus panel size = {panel_size})")


if __name__ == "__main__":
    main()


STAGE 3 (ES vs LS) SMOTE — Feature Selection (10 outer x 3 seeds)
Variants: ['balanced', 'imbalanced']
Class encoding: NEG=LS=0, POS=ES=1
var_prefilter=5000, MI top-K=1000, B=50
Loaded train: 71 samples x 7446 genes
Class balance: {'ES': 41, 'LS': 30}

Phase 1 — zero-variance filter + round to 4 decimals
  Dropped: 0; Kept: 7446

VARIANT: SMOTE = BALANCED
Output dir: stage3_FS_ESLS_smote_balanced
  [seed42_fold1] n_train=63 -> SMOTE: LS 27->36, ES 36->36; var=5000, MI=1000, panel=1000  (30s)
  [seed42_fold2] n_train=64 -> SMOTE: LS 27->37, ES 37->37; var=5000, MI=1000, panel=1000  (23s)
  [seed42_fold3] n_train=64 -> SMOTE: LS 27->37, ES 37->37; var=5000, MI=1000, panel=1000  (24s)
  [seed42_fold4] n_train=64 -> SMOTE: LS 27->37, ES 37->37; var=5000, MI=1000, panel=1000  (24s)
  [seed42_fold5] n_train=64 -> SMOTE: LS 27->37, ES 37->37; var=5000, MI=1000, panel=1000  (26s)
  [seed42_fold6] n_train=64 -> SMOTE: LS 27->37, ES 37->37; var=5000, MI=1000, panel=1000  (23s)
  [seed42_fold7] n